# SP519 · Prédiction du flux d'électrons de 1 MeV en LEO (NOAA-15)

## Step 1 : préparation des données et analyse statistique
Le notebook suit, dans l'ordre, les 7 points du Step 1 de l'énoncé :

1. Récupérer les données NOAA-15 SEM-2 et les paramètres solaires pour 2020–2024
2. Nettoyer et synchroniser les données (valeurs manquantes, valeurs invalides, outliers, horodatage)
3. Choisir la région des ceintures de radiation (plage de L) et justifier ce choix
4. Moyenner tous les paramètres sur des intervalles de 12 h
5. Visualiser les séries temporelles (flux vs temps, L vs temps, paramètres solaires vs temps)
6. Statistiques descriptives, relations entre le flux et les paramètres solaires, outliers
7. Déterminer le seuil du 95ᵉ percentile du flux, qui définit les « événements »

(La cellule « 0 » ci-dessous contient seulement les imports et les paramètres.)

## 0. Imports et paramètres

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:,.4g}')
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'lines.linewidth': 1.0, 'font.size': 9})

# Dossiers (relatifs au notebook)
PROJ = os.getcwd()
DATA = os.path.join(PROJ, 'data')
FIG = os.path.join(PROJ, 'figures')
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIG, exist_ok=True)

# Paramètres de l'étude
START, END = pd.Timestamp('2020-01-01'), pd.Timestamp('2025-01-01')   # période [START, END[
STEP = '12h'            # résolution temporelle des prédictions
L_LO, L_HI = 4.0, 5.5   # région de la ceinture retenue (justifiée au point 3)
P_EVENT = 95            # percentile définissant un "événement"

def savefig(fig, name):
    # Sauvegarde des figures pour le rapport
    fig.savefig(os.path.join(FIG, name), dpi=150, bbox_inches='tight')

print('Dossier du projet :', PROJ)

## 1. Récupération des données (NOAA-15 SEM-2 + paramètres solaires, 2020–2024)

### 1.1 Téléchargement des fichiers bruts NOAA-15
Un fichier par jour sur le serveur NCEI (`poes_n15_AAAAMMJJ_proc.nc`, ~6,6 Mo, une mesure toutes les 2 s), soit
~1 800 fichiers et ~12 Go pour 2020–2024. Ils sont rangés dans `data/noaa15_raw/` (ignoré par Git).
Le téléchargement est **désactivé par défaut** (`TELECHARGER = False`) pour que le notebook puisse s'exécuter de haut
en bas sans retélécharger 12 Go : les tables résumées du point 1.2 sont déjà dans `data/`.

In [ ]:
import calendar
from concurrent.futures import ThreadPoolExecutor
from typing import List

import requests

TELECHARGER = False                          # True pour (re)télécharger les fichiers bruts NOAA-15
RAW_DIR = os.path.join(DATA, 'noaa15_raw')   # fichiers bruts (ignorés par Git)
MAX_WORKERS = 4                              # au-delà, le serveur NCEI bloque des connexions

def download_nc_files(
    years: List[int],
    months: List[int],
    data_dir: str = RAW_DIR,
    max_workers: int = MAX_WORKERS,
) -> None:
    """Downloads NOAA-15 POES daily NetCDF files into data_dir.

    Files already present are skipped (the download can be resumed), and a file
    whose download fails is deleted (no half-written file is left behind).

    Args:
        years: List of years (e.g., [2020, 2021]).
        months: List of months as integers; downloads every day in each month.
        data_dir: Path to the local directory where files will be stored.
        max_workers: Maximum number of files to download concurrently.
    """
    if max_workers < 1:
        raise ValueError("max_workers must be at least 1")

    files_to_download = []
    for year in years:
        for month in months:
            if not 1 <= month <= 12:
                raise ValueError(f"Month must be between 1 and 12, got {month}")
            for day in range(1, calendar.monthrange(year, month)[1] + 1):
                date_str = f"{year:04d}{month:02d}{day:02d}"
                files_to_download.append(
                    (
                        year,
                        date_str,
                        f"poes_n15_{date_str}_proc.nc",
                    )
                )

    # 1. Create the destination directory (existing files are kept)
    os.makedirs(data_dir, exist_ok=True)

    base_url = (
        "https://www.ncei.noaa.gov/data/poes-metop-space-environment-monitor/"
        "access/l1b/v01r00/{year}/noaa15/poes_n15_{date_str}_proc.nc"
    )

    def download_file(file_info):
        year, date_str, filename = file_info
        file_url = base_url.format(year=year, date_str=date_str)
        destination_path = os.path.join(data_dir, filename)
        if os.path.exists(destination_path):
            return                                   # already downloaded
        tmp_path = destination_path + ".part"        # renamed only once complete

        print(f"Downloading: {file_url}")
        try:
            with requests.get(file_url, stream=True, timeout=30) as response:
                response.raise_for_status()

                with open(tmp_path, "wb") as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)

            os.replace(tmp_path, destination_path)
            print(f"Successfully saved to {destination_path}")

        except requests.exceptions.RequestException as e:
            if os.path.exists(tmp_path):
                os.remove(tmp_path)
            print(f"Failed to download {file_url}: {e}")

    # 2. Download daily files concurrently to reduce time spent waiting on the server.
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        list(executor.map(download_file, files_to_download))


if TELECHARGER:
    download_nc_files(years=list(range(START.year, END.year)), months=list(range(1, 13)))
else:
    print('Téléchargement désactivé (TELECHARGER = False) : on utilise les tables déjà présentes dans data/.')

### 1.2 Paramètres solaires et géomagnétiques (OMNI)
Base **OMNI2 horaire** (`OMNI2_H0_MRG1HR`) de la NASA (SPDF / CDAWeb) : mesures du vent solaire (sondes ACE, Wind…)
ramenées jusqu'à la Terre, et indices géomagnétiques, sur une même base de temps horaire UTC. Deux fichiers CDF par an
(janvier–juin, juillet–décembre), soit 10 fichiers d'environ 1,5 Mo, lus avec `cdflib`.

| Variable OMNI | Nom dans le notebook | Rôle physique |
|---|---|---|
| `V` | `V_kms` | vitesse du vent solaire : principal moteur de l'accélération des électrons relativistes |
| `BZ_GSM` | `Bz_nT` | Bz < 0 → reconnexion magnétique, injection de particules dans la magnétosphère |
| `F10_INDEX` | `F107_sfu` | flux radio solaire à 10,7 cm : niveau d'activité du cycle solaire |
| `N`, `Pressure` | `N_cm3`, `P_nPa` | densité et pression dynamique : compression de la magnétosphère, pertes d'électrons |
| `E` | `E_mVm` | champ électrique −V·Bz : couplage vent solaire / magnétosphère |
| `ABS_B` | `B_nT` | intensité du champ magnétique interplanétaire |
| `KP`, `DST`, `AE` | `Kp`, `Dst_nT`, `AE_nT` | activité géomagnétique : globale (Kp), orages (Dst), sous-orages (AE) |

V, Bz et F10.7 sont demandés par l'énoncé ; les autres sont ajoutés pour les raisons physiques ci-dessus.
Les valeurs manquantes sont codées dans les CDF par une valeur de remplissage (attribut `FILLVAL`, ex. 9999 pour V) :
on les remplace par NaN. `KP` est stocké ×10 (Kp = 3+ → 33) : on le divise par 10.
Le flux de protons > 10 MeV (`PR-FLX_10`) n'est pas gardé : il manque à ~97 % sur 2020–2024.

In [ ]:
import cdflib

OMNI_DIR = os.path.join(DATA, 'omni')   # fichiers CDF bruts (ignorés par Git)
OMNI_URL = 'https://spdf.gsfc.nasa.gov/pub/data/omni/omni_cdaweb/hourly/{y}/omni2_h0_mrg1hr_{y}{m}01_v01.cdf'
OMNI_VARS = {'V': 'V_kms', 'BZ_GSM': 'Bz_nT', 'F10_INDEX': 'F107_sfu', 'N': 'N_cm3', 'Pressure': 'P_nPa',
             'E': 'E_mVm', 'ABS_B': 'B_nT', 'KP': 'Kp', 'DST': 'Dst_nT', 'AE': 'AE_nT'}
os.makedirs(OMNI_DIR, exist_ok=True)

morceaux = []
for y in range(START.year, END.year):
    for m in ('01', '07'):                                   # un fichier par semestre
        path = os.path.join(OMNI_DIR, f'omni2_h0_mrg1hr_{y}{m}01_v01.cdf')
        if not os.path.exists(path):                         # téléchargé une seule fois
            r = requests.get(OMNI_URL.format(y=y, m=m), timeout=60)
            r.raise_for_status()
            with open(path, 'wb') as f:
                f.write(r.content)
        cdf = cdflib.CDF(path)
        d = {'time': cdflib.cdfepoch.to_datetime(cdf.varget('Epoch'))}
        for var, nom in OMNI_VARS.items():
            x = cdf.varget(var).astype(float)
            fill = cdf.varattsget(var)['FILLVAL']
            x[np.isclose(x, fill) | (x >= fill)] = np.nan       # valeur de remplissage -> NaN
            d[nom] = x
        morceaux.append(pd.DataFrame(d))

omni_raw = pd.concat(morceaux).drop_duplicates('time').set_index('time').sort_index()
omni_raw = omni_raw.loc[START:END - pd.Timedelta('1h')]
omni_raw['Kp'] = omni_raw['Kp'] / 10                         # Kp*10 -> Kp
omni_raw.index = pd.DatetimeIndex(omni_raw.index).as_unit('ns')
omni_raw.to_parquet(os.path.join(DATA, 'omni_hourly_2020_2024.parquet'))
print('OMNI :', omni_raw.shape, '|', omni_raw.index.min(), '->', omni_raw.index.max())

### 1.3 Tables utilisées dans la suite du notebook
La cellule ci-dessous lit les deux tables de `data/` ; tout le reste du notebook ne dépend que d'elles.

* `noaa15_p6_hourly_Lbins.parquet` : le flux `mep_pro_tel90_flux_p6` de NOAA-15, résumé **heure par heure et par
  tranche de L** (L de 1 à 8 par pas de 0,25, et `Lbin = 10` pour la calotte polaire L ≥ 10).
  Colonnes : `time` (début de l'heure, UTC), `Lbin`, `p6_sum` (somme des flux), `n` (nombre de mesures),
  `p6_max` (plus grande mesure de 2 s de la case, utile pour repérer les pics aberrants),
  `omni_p1_sum` / `omni_p1_n` (idem pour les protons 25 MeV, utiles au point 2).
  → Produite au **point 2** à partir des fichiers bruts (nettoyage puis résumé) : **en cours, autre membre du
  binôme**. En attendant, la version présente dans `data/` a été produite selon le même principe.
* `omni_hourly_2020_2024.parquet` : paramètres OMNI heure par heure (index `time`, UTC) : `V_kms`, `Bz_nT`,
  `F107_sfu`, `N_cm3`, `P_nPa`, `E_mVm`, `B_nT`, `Kp`, `Dst_nT`, `AE_nT`.
  → Produite par la cellule OMNI ci-dessus (1.2).

In [ ]:
noaa = pd.read_parquet(os.path.join(DATA, 'noaa15_p6_hourly_Lbins.parquet'))
omni = pd.read_parquet(os.path.join(DATA, 'omni_hourly_2020_2024.parquet'))

L_POLAR = 10.0   # code de la tranche "calotte polaire" (L >= 10)

print('NOAA-15 :', noaa.shape, '|', noaa.time.min(), '->', noaa.time.max())
print('   tranches de L :', [float(l) for l in sorted(noaa.Lbin.unique())])
jours = noaa.time.dt.floor('D').unique()
print('   jours avec données :', len(jours), '/', (END - START).days)
print('OMNI    :', omni.shape, '|', omni.index.min(), '->', omni.index.max())
print('   % manquant :', omni.isna().mean().mul(100).round(2).to_dict())
noaa.head()

## 2. Nettoyage et synchronisation

> **En cours : écrit par l'autre membre du binôme** (valeurs manquantes / invalides, contamination par les protons
> solaires, horodatage). Sera intégré ici.

## 3. Choix de la région des ceintures de radiation (plage de L)

**Le paramètre L** : distance (en rayons terrestres) à laquelle la ligne de champ magnétique sur laquelle se trouve
le satellite croise l'équateur magnétique. NOAA-15 est en orbite basse (~810 km) mais, en passant d'un pôle à
l'autre, il traverse toutes les lignes de champ, donc toutes les valeurs de L. Le flux mesuré dépend énormément de L :
il faut choisir **quelle région** on cherche à prédire.

**Ce que dit la physique :**
* **L < 2 : ceinture interne.** Peuplée de protons piégés très énergétiques (Anomalie Sud-Atlantique) : P6 y compte
  surtout des **protons**, pas des électrons → à exclure.
* **L ≈ 2–3 : zone « slot ».** Quasi vide d'électrons, sauf après de très forts orages.
* **L ≈ 3–7 : ceinture externe.** C'est là que se trouvent les électrons relativistes (~1 MeV) qui varient fortement
  avec le vent solaire : c'est la région qui nous intéresse.
* **L > 7 :** on sort des ceintures (lignes de champ ouvertes ou très étirées) → flux au niveau du bruit.

**Critères de choix**, vérifiés sur les données 2020–2024 ci-dessous. La région retenue doit :
1. avoir un flux **nettement au-dessus du bruit de fond** de l'instrument (sinon on prédit du bruit) ;
2. être **variable** dans le temps (sinon il n'y a rien à prédire) ;
3. être **liée au vent solaire** (sinon les paramètres OMNI ne serviront à rien pour prédire).

### 3.1 Carte L – temps du flux
Pour chaque jour et chaque tranche de L, flux moyen = Σ flux / Σ mesures. L'échelle de couleur est logarithmique
(le flux varie sur plusieurs ordres de grandeur). L'indice Dst est tracé en dessous : les fortes valeurs négatives
signalent les orages géomagnétiques.

In [ ]:
# Niveau de bruit : dans la calotte polaire (L >= 10) il n'y a pas d'électrons piégés,
# P6 n'y mesure que le bruit de fond de l'instrument
pc = noaa[noaa.Lbin == L_POLAR]
NOISE = (pc.p6_sum / pc.n).median()
print(f'Bruit de fond P6 (médiane horaire en calotte polaire) : {NOISE:.0f} #/cm²/s/sr')

# Moyenne journalière par tranche de L (L < 8)
belt = noaa[noaa.Lbin < L_POLAR]
d = belt.groupby([belt.time.dt.floor('D'), 'Lbin'])[['p6_sum', 'n']].sum()
carte = (d.p6_sum / d.n).unstack('Lbin')
carte = carte.reindex(pd.date_range(START, END, freq='D', inclusive='left'))   # jours manquants -> NaN

fig, ax = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
cmap = plt.get_cmap('viridis').copy()
cmap.set_bad('0.85')                                      # jours manquants en gris
im = ax[0].imshow(np.log10(carte.T.clip(lower=NOISE / 2)).values, aspect='auto', origin='lower', cmap=cmap,
                  extent=[mdates.date2num(START), mdates.date2num(END), 1, 8], interpolation='nearest')
for l in (L_LO, L_HI):
    ax[0].axhline(l, color='w', ls='--', lw=1)
ax[0].grid(False)
ax[0].set_ylabel('L (IGRF)')
ax[0].set_title('NOAA-15 P6 (électrons ~1 MeV) : flux moyen journalier par tranche de L, 2020–2024')
fig.colorbar(im, ax=ax, label='log10 flux P6 (#/cm²/s/sr)', pad=0.01, fraction=0.03)
ax[1].plot(omni.index, omni['Dst_nT'], color='C0', lw=0.5)
ax[1].set_ylabel('Dst (nT)')
ax[1].xaxis_date()
savefig(fig, '3_carte_L_temps.png')
plt.show()

### 3.2 Profil en L : niveau, variabilité et lien avec le vent solaire
On calcule, pour chaque tranche de L, le flux moyen sur des fenêtres de 12 h (la résolution du projet), puis :
* la **médiane** et les percentiles 5 % / 95 % du log10 du flux → niveau par rapport au bruit (critère 1) ;
* le **% de fenêtres où le flux dépasse 2 × le bruit** (critère 1) ;
* l'**écart-type du log10 du flux** → variabilité (critère 2) ;
* la **corrélation de Spearman** entre le log10 du flux et la vitesse du vent solaire V **24 h plus tôt**
  (le vent solaire met 1 à 2 jours à accélérer les électrons) → lien avec le vent solaire (critère 3).

In [ ]:
d12 = belt.groupby([pd.Grouper(key='time', freq=STEP), 'Lbin'])[['p6_sum', 'n']].sum()
f12 = (d12.p6_sum / d12.n).unstack('Lbin')                # flux 12 h : une colonne par tranche de L
logf = np.log10(f12.clip(lower=NOISE / 2))
V12 = omni['V_kms'].resample(STEP).mean()

profil = pd.DataFrame({
    'log10 P5': logf.quantile(0.05),
    'log10 médiane': logf.median(),
    'log10 P95': logf.quantile(0.95),
    '% > 2×bruit': (f12 > 2 * NOISE).mean() * 100,
    'écart-type log10': logf.std(),
    'Spearman V(t-24h)': logf.apply(lambda c: c.corr(V12.shift(2).reindex(c.index), method='spearman')),
})
profil.index.name = 'L (bord gauche)'
display(profil.round(2))

x = profil.index + 0.125                                  # centre des tranches
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
ax[0].fill_between(x, profil['log10 P5'], profil['log10 P95'], color='C0', alpha=0.25, label='P5 – P95')
ax[0].plot(x, profil['log10 médiane'], color='C0', lw=2, label='médiane')
ax[0].axhline(np.log10(NOISE), color='C3', ls='--', label='bruit')
ax[0].set_ylabel('log10 flux (12 h)')
ax[0].set_title('1. Niveau')
ax[0].legend(fontsize=7)
ax[1].plot(x, profil['écart-type log10'], color='C0', lw=2)
ax[1].set_ylabel('écart-type de log10 flux')
ax[1].set_title('2. Variabilité')
ax[2].plot(x, profil['Spearman V(t-24h)'], color='C0', lw=2)
ax[2].set_ylabel('ρ Spearman')
ax[2].set_title('3. Lien avec V 24 h avant')
for a in ax:
    a.axvspan(L_LO, L_HI, color='C1', alpha=0.2)
    a.set_xlabel('L')
fig.suptitle(f'Profil en L (fenêtres de 12 h, 2020–2024) — zone orange : région retenue {L_LO} ≤ L < {L_HI}')
fig.tight_layout()
savefig(fig, '3_profil_L.png')
plt.show()

### 3.3 Conclusion : région retenue **4 ≤ L < 5,5**

**Ce que montre la carte L – temps (3.1) :**
* une bande **constante à L ≈ 1,25–1,75** : les protons de la ceinture interne, insensibles aux orages → exclue ;
* une **zone « slot » sombre à L ≈ 2–3**, remplie seulement lors des orages les plus forts (2023, mai 2024 :
  Dst ≈ −400 nT), quand la ceinture externe est poussée vers la Terre ;
* une **ceinture externe à L ≈ 3,5–6** qui s'allume et s'éteint au rythme de l'activité solaire : c'est la cible ;
* au-delà de L ≈ 7, le flux reste au niveau du bruit.

**Ce que montre le profil en L (3.2), tranche par tranche :**

| Critère | 3,5 ≤ L < 4 | **4 ≤ L < 5,5** | 5,5 ≤ L < 6 |
|---|---|---|---|
| 1. Fenêtres de 12 h au-dessus de 2 × bruit | 68–80 % | **82–94 %** | 60–72 % |
| 2. Variabilité (écart-type de log10 flux) | 0,68 | **0,46–0,66** | 0,38–0,42 |
| 3. Corrélation avec V 24 h avant | 0,31–0,32 | **0,30–0,32** | 0,27–0,28 |

La plage **4 ≤ L < 5,5** est la seule qui remplit les trois critères à la fois : le flux y est **au-dessus du
bruit dans plus de 80 % des fenêtres** (c'est aussi là que se trouve le maximum de flux, vers L ≈ 4,25–4,5), il
**varie fortement** (on a quelque chose à prédire), et il est **lié au vent solaire**. En dessous de L = 4, la
variabilité est plus forte mais le flux tombe trop souvent dans le bruit (et la région n'est peuplée qu'après les
orages) ; au-dessus de L = 5,5, le flux est trop souvent au niveau du bruit et moins variable.

Limite : la ceinture se déplace (vers la Terre pendant les forts orages). Une plage fixe ne la suit pas toujours ;
comparer plusieurs plages de L est d'ailleurs proposé en bonus par l'énoncé.